# Sourmash Profiling Of All Clean Reads

One shared report for QC, viral metagenomes and phage isolates. Each sample contributes all final clean paired and orphan reads. Their abundance sketches are merged into one pool, including any negative control. This is a report-only GTDB bacterial/archaeal screen; no reads or contigs are removed.


In [ ]:
from pathlib import Path
from html import escape
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SAMPLES = list(snakemake.params.samples)
negative_control = str(snakemake.params.negative_control)
minimum_shared_bp = snakemake.params.minimum_shared_bp
minimum_reference_fraction = snakemake.params.minimum_reference_fraction
database = str(snakemake.params.database)
for path in snakemake.output:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
sns.set_context("notebook")
plt.rcParams["svg.fonttype"] = "none"
display(pd.DataFrame([
    {"parameter": "database", "value": database},
    {"parameter": "minimum_shared_bp", "value": minimum_shared_bp},
    {"parameter": "minimum_reference_kmer_fraction", "value": minimum_reference_fraction},
    {"parameter": "negative_control", "value": negative_control or "not specified"},
]))


## Reference Evidence And Profile Summary

A supported reference has at least the configured estimated shared bases and reference k-mer containment. Weaker returned matches remain in the tables and plots. Shared bases are sketch estimates; containment and matching k-mer abundance are not mapping breadth or sequencing depth. Distinct species labels are database assignments, not confirmed species identifications. Every sample remains in the summary, including zero-hit samples.


In [ ]:
def build_clean_read_profiles(paths, minimum_shared_bp, minimum_reference_fraction, database):
    tables = []
    for path in paths:
        frame = pd.read_csv(path).rename(columns={"match_name": "reference_name", "name": "reference_name"})
        frame["sample"] = Path(path).name.removesuffix("_gather_sourmash.with-lineages.csv")
        frame["profile_scope"] = "pooled" if Path(path).parent.name == "POOLED" else "sample"
        for field in ["unique_intersect_bp", "scaled", "f_match_orig", "f_unique_weighted", "average_abund", "median_abund"]:
            frame[field] = pd.to_numeric(frame.get(field, pd.Series(index=frame.index, dtype=float)), errors="coerce")
        frame["reference_accession"] = frame["reference_name"].astype(str).str.split().str[0]
        frame["lineage"] = frame.get("lineage", pd.Series(index=frame.index, dtype=str)).fillna("")
        lineage = frame["lineage"].str.split(";")
        for rank, position in [("superkingdom", 0), ("family", 4), ("genus", 5), ("species", 6)]:
            frame[rank] = lineage.str[position].fillna("").astype(str).str.replace(r"^[a-z]__", "", regex=True).replace("", "not reported")
        frame["shared_bp"] = frame["unique_intersect_bp"]
        frame["shared_hashes"] = frame["unique_intersect_bp"] / frame["scaled"].replace(0, np.nan)
        frame["reference_kmer_fraction"] = frame["f_match_orig"]
        frame["reference_kmer_percent"] = 100 * frame["f_match_orig"]
        frame["weighted_query_fraction"] = frame["f_unique_weighted"]
        frame["mean_kmer_abundance"] = frame["average_abund"]
        frame["median_kmer_abundance"] = frame["median_abund"]
        frame["passes_profile_filter"] = (frame["shared_bp"] >= minimum_shared_bp) & (frame["reference_kmer_fraction"] >= minimum_reference_fraction)
        frame["evidence_class"] = np.where(frame["passes_profile_filter"], "supported", "weak")
        frame["database"] = database
        frame["evidence_scope"] = "final_clean_reads"
        tables.append(frame)
    columns = ["profile_scope", "sample", "reference_accession", "reference_name", "lineage",
               "superkingdom", "family", "genus", "species", "shared_bp", "shared_hashes",
               "reference_kmer_fraction", "reference_kmer_percent", "weighted_query_fraction",
               "mean_kmer_abundance", "median_kmer_abundance", "passes_profile_filter",
               "evidence_class", "database", "evidence_scope"]
    return pd.concat(tables, ignore_index=True).reindex(columns=columns).fillna("not reported")


clean_read_profiles = build_clean_read_profiles(
    snakemake.input.gather, minimum_shared_bp, minimum_reference_fraction, database,
)
taxonomy_tables = []
for path in snakemake.input.profiles:
    frame = pd.read_csv(path)
    frame["sample"] = Path(path).name.removesuffix("_sourmash.summarized.csv")
    frame["profile_scope"] = "pooled" if Path(path).parent.name == "POOLED" else "sample"
    taxonomy_tables.append(frame)
clean_read_taxonomy = pd.concat(taxonomy_tables, ignore_index=True).reindex(columns=[
    "profile_scope", "sample", "rank", "lineage", "fraction", "f_weighted_at_rank", "bp_match_at_rank",
])

statuses = []
for path in snakemake.input.gather:
    sample = Path(path).name.removesuffix("_gather_sourmash.with-lineages.csv")
    scope = "pooled" if Path(path).parent.name == "POOLED" else "sample"
    matches = clean_read_profiles[clean_read_profiles["sample"].eq(sample) & clean_read_profiles["profile_scope"].eq(scope)]
    supported = matches.loc[matches["passes_profile_filter"].eq(True)]
    statuses.append({
        "sample": sample, "profile_scope": scope,
        "is_negative_control": scope == "sample" and sample == negative_control,
        "returned_references": len(matches),
        "supported_references": len(supported),
        "weak_references": len(matches) - len(supported),
        "reported_species": matches.loc[matches["species"].ne("not reported"), "species"].nunique(),
        "supported_species": supported.loc[supported["species"].ne("not reported"), "species"].nunique(),
        "matched_weighted_query_fraction": pd.to_numeric(matches["weighted_query_fraction"], errors="coerce").sum(),
    })
clean_profile_status = pd.DataFrame(statuses)
clean_read_profiles.to_csv(snakemake.output.metadata, sep="\t", index=False)
clean_read_taxonomy.to_csv(snakemake.output.taxonomy, sep="\t", index=False)
clean_profile_status.to_csv(snakemake.output.status, sep="\t", index=False)
display(clean_profile_status)
display(clean_read_profiles.head(30))
display(clean_read_taxonomy.head(30))


## Pooled Genus Profile

The pool is weighted by each library's sequencing contribution, not an equal-weight average. The plot shows up to 30 genera; all returned references remain in the exported tables. A negative control remains in the pool and is not subtracted.


In [ ]:
# All returned matches are kept; the support filter labels evidence rather than removing it.
genus_profiles = clean_read_profiles.copy()
genus_profiles["weighted_query_fraction"] = pd.to_numeric(genus_profiles["weighted_query_fraction"], errors="coerce").fillna(0)
genus_profiles["genus_lineage"] = genus_profiles["lineage"].astype(str).str.split(";").str[:6].str.join(";")
genus_profiles["genus_label"] = genus_profiles["genus"].where(genus_profiles["genus"].ne("not reported"), "Unassigned genus")
pooled = genus_profiles[genus_profiles["profile_scope"].eq("pooled")]
pooled_genus = pooled.groupby(["genus_lineage", "genus_label"], dropna=False)["weighted_query_fraction"].sum().reset_index()
pooled_genus = pooled_genus.sort_values("weighted_query_fraction", ascending=False).head(30)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(pooled_genus) + 1.5)))
if pooled_genus.empty:
    ax.text(0.5, 0.5, "No GTDB matches above the gather threshold", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
else:
    ax.barh(np.arange(len(pooled_genus)), 100 * pooled_genus["weighted_query_fraction"], color="#3A72ED")
    ax.set_yticks(np.arange(len(pooled_genus)), labels=pooled_genus["genus_label"])
    ax.invert_yaxis()
    ax.set_xlabel("Abundance-weighted query fraction (%)")
    sns.despine(ax=ax)
ax.set_title("Pooled clean-read GTDB profile (all samples)")
fig.tight_layout()
fig.savefig(snakemake.output.pooled_png, dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.pooled_svg, bbox_inches="tight")
plt.show()
plt.close(fig)


## Per-Sample Genus Profiles

Colour is log-transformed abundance-weighted query percentage. A star means at least one reference assigned to that genus passes both support criteria. All returned matches contribute, including weak matches; blank stars do not mean absence. Samples follow workflow order, with at most 50 per row. A specified negative control has a red sample label. The full, untransformed genus percentage matrix is exported separately.


In [ ]:
per_sample = genus_profiles[genus_profiles["profile_scope"].eq("sample")]
matrix = per_sample.pivot_table(index="genus_lineage", columns="sample", values="weighted_query_fraction", aggfunc="sum", fill_value=0)
matrix = matrix.reindex(columns=SAMPLES, fill_value=0)
(100 * matrix).to_csv(snakemake.output.genus_matrix, sep="\t")
selected = matrix.max(axis=1).sort_values(ascending=False).head(30).index
plot_matrix = matrix.reindex(index=selected)
support = per_sample.pivot_table(index="genus_lineage", columns="sample", values="passes_profile_filter", aggfunc="max", fill_value=False)
support = support.reindex(index=plot_matrix.index, columns=plot_matrix.columns, fill_value=False)
labels = genus_profiles.drop_duplicates("genus_lineage").set_index("genus_lineage")["genus_label"]

# Maximum 50 samples per row; the final row keeps the same sample-cell width.
columns_per_row = min(50, max(1, len(SAMPLES)))
rows = max(1, int(np.ceil(len(SAMPLES) / columns_per_row)))
fig, axes = plt.subplots(rows, 1, squeeze=False,
    figsize=(max(8, 0.35 * columns_per_row + 3), rows * max(4, 0.35 * len(plot_matrix) + 2)))
maximum = max(1.0, float(np.log10(1 + 100 * plot_matrix).max().max())) if not plot_matrix.empty else 1.0
for row, ax in enumerate(axes[:, 0]):
    sample_page = SAMPLES[row * columns_per_row:(row + 1) * columns_per_row]
    if plot_matrix.empty:
        ax.text(0.5, 0.5, "No per-sample GTDB matches above the gather threshold", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    else:
        stars = support[sample_page].map(lambda value: "*" if value else "")
        sns.heatmap(np.log10(1 + 100 * plot_matrix[sample_page]), cmap="mako", annot=stars, fmt="",
                    vmin=0, vmax=maximum, yticklabels=labels.reindex(plot_matrix.index).tolist(), ax=ax,
                    cbar_kws={"label": "log10(1 + weighted query %)"})
        ax.set_xlim(0, columns_per_row)
        ax.set_xlabel("Sample")
        ax.set_ylabel("GTDB genus")
        ax.tick_params(axis="x", rotation=90)
        for tick in ax.get_xticklabels():
            if negative_control and tick.get_text() == negative_control:
                tick.set_color("#b42318")
                tick.set_fontweight("bold")
    ax.set_title(f"Clean-read profiles: * = supported reference; sample page {row + 1}/{rows}")
fig.tight_layout()
fig.savefig(snakemake.output.genus_png, dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.genus_svg, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
notes = (
    "Full final-clean paired and orphan reads; no BBNorm-normalised reads or 2M subset. "
    "GTDB RS226 screens bacteria and archaea, not viruses or every potential contaminant. "
    "Supported references meet both the shared-base and reference-k-mer-containment thresholds; weaker returned matches remain visible. "
    "Reference k-mer containment is not mapping breadth; matching k-mer abundance is not sequencing depth. "
    "Species counts describe distinct assigned database labels, not confirmed species identifications. "
    "Weighted query fractions describe sampled k-mer abundance, not an exact fraction of sequencing reads. "
    "The pool includes every sample and any negative control, weighted by actual sequencing contribution. "
    "The control is labelled but is not subtracted or tested for enrichment. "
    "No matches above the gather threshold does not mean that a library is contamination-free."
)
html = "<!doctype html><html><head><meta charset='utf-8'><title>MOSAIC clean-read Sourmash</title>"
html += "<style>body{font-family:Arial,sans-serif;margin:2em}table{border-collapse:collapse}th,td{padding:0.4em;text-align:left}img{max-width:100%}</style></head><body>"
html += "<h1>Sourmash profiling of all final clean reads</h1><p>" + escape(notes) + "</p>"
html += f"<p>Database: {escape(database)}. Support: ≥{minimum_shared_bp:,} estimated shared bp and ≥{100 * minimum_reference_fraction:g}% reference k-mer containment.</p>"
html += "<h2>Profile summary</h2>" + clean_profile_status.to_html(index=False)
html += "<p>Complete exported tables: "
for name, title in [("metadata", "Reference evidence"), ("taxonomy", "Taxonomic profiles"),
                    ("status", "Sample and pooled summary"), ("genus_matrix", "Genus query percentage matrix")]:
    html += f"<a href='{escape(Path(getattr(snakemake.output, name)).name)}'>{title}</a> · "
html += "</p>"
for title, path in [("Pooled genus profile", snakemake.output.pooled_png),
                    ("Per-sample genus profiles (* = supported reference)", snakemake.output.genus_png)]:
    encoded = base64.b64encode(Path(path).read_bytes()).decode()
    html += f"<h2>{title}</h2><img src='data:image/png;base64,{encoded}'>"
html += "<h2>Reference evidence: first 30 rows</h2>" + clean_read_profiles.head(30).to_html(index=False)
html += "<h2>Taxonomic profiles: first 30 rows</h2>" + clean_read_taxonomy.head(30).to_html(index=False)
html += "</body></html>"
Path(snakemake.output.html).write_text(html, encoding="utf-8")
